# HyperKT Objective 01 — Stage 2 (corrected v2)
**Runtime:** CPU / Python 3. Corrected standardized columns and Stage 1 unknown/padding IDs.


In [ ]:
!pip -q install pandas numpy pyarrow scipy

In [ ]:
from google.colab import files
uploaded = files.upload()  # select hyperkt_stage1.zip
assert 'hyperkt_stage1.zip' in uploaded, 'Upload hyperkt_stage1.zip'


In [ ]:
import os, json, zipfile, hashlib, platform
from pathlib import Path
import numpy as np, pandas as pd
ROOT=Path('/content/hyperkt_stage2'); ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile('/content/hyperkt_stage1.zip') as z:
    required={'manifest.json','skill_vocab.json','train_interactions.parquet','val_interactions.parquet','test_interactions.parquet'}
    assert required.issubset(z.namelist()), f'Missing: {required-set(z.namelist())}'
    z.extractall(ROOT)
manifest=json.loads((ROOT/'manifest.json').read_text())
vocab=json.loads((ROOT/'skill_vocab.json').read_text())
print('Python:',platform.python_version(),'Stage1 SHA256:',hashlib.sha256(Path('/content/hyperkt_stage1.zip').read_bytes()).hexdigest())
print('Stage1 clean rows:',manifest['clean_rows'],'vocab entries:',len(vocab))

In [ ]:
frames={s:pd.read_parquet(ROOT/f'{s}_interactions.parquet') for s in ('train','val','test')}
for s,df in frames.items():
    print(s,df.shape,df.columns.tolist(), '\n',df.head(2).to_string(index=False))


## Column mapping
Stage 1 stores original column names in its manifest. We check them against the actual Parquet schema and refuse to silently guess labels.

In [ ]:
# Stage 1 exported standardized Parquet columns; manifest['columns'] records ORIGINAL CSV names.
USER, SKILL, CORRECT, ORDER = 'user', 'skill', 'correct', 'order'
for name, df in frames.items():
    required={USER,SKILL,CORRECT,ORDER}
    assert required.issubset(df.columns), f'{name}: missing {required-set(df.columns)}; actual {df.columns.tolist()}'
    assert df[[USER,SKILL,CORRECT,ORDER]].notna().all().all(), f'{name}: null data'
    assert df[CORRECT].isin([0,1]).all(), f'{name}: non-binary correctness'
    print(f'{name}: PASS standardized columns, no nulls, binary responses')


## Sequence protocol
Stage 1 reserves skill ID **0 for unknown** and **1 for padding**; vocabulary IDs start at 2. These IDs are preserved throughout Stage 2. Targets 1..T-1 are partitioned into disjoint blocks of at most 200. Each target uses the immediately previous interaction as input, with causal attention required in Stage 3. The first target of each block does not have the full preceding history; this is a documented context limitation.


In [ ]:
MAX_LEN=200
assert MAX_LEN>=2
# Stage1 vocabulary maps original skill labels to integer IDs, with 0 reserved for UNK and 1 for PAD.
lookup={str(k):int(v) for k,v in vocab.items()}
assert 0 not in lookup.values() and 1 not in lookup.values()

def norm_skill(v):
    # preserve composite skill IDs as strings; avoid accidentally altering original labels
    return str(v).strip()

def build_windows(df):
    Xs=[]; Xr=[]; Qs=[]; Ys=[]; Ms=[]; U=[]
    # stable sort preserves original row order for tied order IDs
    ordered=df.sort_values([USER,ORDER],kind='mergesort')
    for uid,g in ordered.groupby(USER,sort=False):
        skills=np.asarray([lookup.get(norm_skill(x),0) for x in g[SKILL].to_numpy()],dtype=np.int32)
        answers=g[CORRECT].to_numpy(dtype=np.int8)
        if len(skills)<2: continue
        # Partition target indices 1..T-1 into disjoint blocks of <= MAX_LEN.
        # Each window uses immediately preceding interactions as its aligned inputs.
        for start in range(1,len(skills),MAX_LEN):
            end=min(start+MAX_LEN,len(skills)); L=end-start
            x_skill=np.full(MAX_LEN,1,dtype=np.int32)
            x_resp=np.zeros(MAX_LEN,dtype=np.int8)
            q_skill=np.full(MAX_LEN,1,dtype=np.int32)
            y=np.zeros(MAX_LEN,dtype=np.int8)
            mask=np.zeros(MAX_LEN,dtype=np.bool_)
            x_skill[:L]=skills[start-1:end-1]
            x_resp[:L]=answers[start-1:end-1]
            q_skill[:L]=skills[start:end]
            y[:L]=answers[start:end]
            mask[:L]=True
            Xs.append(x_skill); Xr.append(x_resp); Qs.append(q_skill)
            Ys.append(y); Ms.append(mask); U.append(str(uid))
    if not Xs: raise ValueError('No valid sequences: check source data')
    return {'history_skill':np.stack(Xs),'history_correct':np.stack(Xr),
            'target_skill':np.stack(Qs),'target_correct':np.stack(Ys),
            'mask':np.stack(Ms),'user_id':np.asarray(U,dtype=str)}

arrays={s:build_windows(df) for s,df in frames.items()}
for s,a in arrays.items():
    np.savez_compressed(ROOT/f'{s}_sequences.npz',**a)
    print(s,'windows',len(a['mask']),'targets',int(a['mask'].sum()),'unknown target skills',int(((a['target_skill']==0)&a['mask']).sum()))


## Mandatory integrity checks
These tests detect split overlap, incorrect label shifts, repeated target coverage and padding mistakes. The training code in Stage 3 must apply causal attention and must not feed `target_correct` into the input.

In [ ]:
student_sets={s:set(map(str,df[USER].unique())) for s,df in frames.items()}
assert not(student_sets['train']&student_sets['val'])
assert not(student_sets['train']&student_sets['test'])
assert not(student_sets['val']&student_sets['test'])
for s,a in arrays.items():
    expected=int(frames[s].groupby(USER).size().sub(1).clip(lower=0).sum())
    actual=int(a['mask'].sum())
    assert actual==expected,(s,actual,expected)
    m=a['mask']; assert np.all(a['target_skill'][~m]==1)
    assert np.all(a['history_skill'][~m]==1)
    assert np.all(a['target_correct'][m]<=1)
    assert np.all(a['history_correct'][m]<=1)
    assert np.all(a['history_skill'][m]!=1)
    assert np.all(a['target_skill'][m]!=1)
    print(f'{s}: PASS — {actual:,} uniquely scored next-response targets; {len(student_sets[s]):,} students')
print('PASS: student-disjoint splits and next-response target coverage')


In [ ]:
report={'stage':2,'max_len':MAX_LEN,'protocol':'disjoint target blocks, immediate previous-response inputs',
 'source_sha256':hashlib.sha256(Path('/content/hyperkt_stage1.zip').read_bytes()).hexdigest(),
 'splits':{s:{'students':len(student_sets[s]),'windows':int(len(arrays[s]['mask'])),
               'targets':int(arrays[s]['mask'].sum()),'unknown_targets':int(((arrays[s]['target_skill']==0)&arrays[s]['mask']).sum())}
          for s in arrays}}
(ROOT/'stage2_report.json').write_text(json.dumps(report,indent=2))
print(json.dumps(report,indent=2))


In [ ]:
# Export only the Stage 2 outputs, plus the Stage 1 metadata and co-occurrence matrix needed downstream.
keep=['stage2_report.json','manifest.json','skill_vocab.json','train_cooccurrence.npz']+[f'{s}_sequences.npz' for s in ('train','val','test')]
output=Path('/content/hyperkt_stage2.zip')
with zipfile.ZipFile(output,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for name in keep:
        p=ROOT/name
        if p.exists():z.write(p,arcname=name)
print('Saved:',output,'bytes:',output.stat().st_size)
files.download(str(output))


### Before Stage 3
Upload `hyperkt_stage2.zip` here and share the three `PASS` lines plus the report. **Research note:** This sequence construction uses a one-step input for the first token of each disjoint target block, rather than full 200-step historical context across block boundaries. Stage 3 should explicitly document that choice, or revise it to use longer left context without duplicating targets.